<a href="https://colab.research.google.com/github/galvanoski/TurungInterviewApp/blob/main/tool_calling_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tool Calling Evaluation

In this notebook, you will build an evaluation pipeline for tool calling — a critical component of LLM-powered agents. When an agent receives a user query, it must decide *which* tool to call and *what parameters* to pass. Getting either wrong can produce incorrect results.

You will evaluate a career consultant agent using two complementary approaches:
- **Deterministic checks** — fast, free string comparison for tool selection and parameter accuracy
- **LLM-as-a-judge (DiscreteMetric)** — nuanced evaluation with written reasoning

---

### Learning Objectives

By the end of this notebook, you will be able to:

1. Design test datasets for tool calling evaluation with varied difficulty levels
2. Evaluate tool selection and parameter accuracy with deterministic checks
3. Use RAGAs `DiscreteMetric` as an LLM-as-a-judge for nuanced tool calling evaluation
4. Compare deterministic vs. LLM-based evaluation to understand where each excels

### Estimated Cost

This notebook makes API calls via OpenRouter. **Estimated total cost: ~$ 0.05 – 0.15** depending on response lengths and retries.

In [ ]:
%pip install openai langchain==1.2.17 langchain-openai==1.2.1 langchain-core==1.3.2 ragas==0.4.3 pandas -q

In [ ]:
import os

from google.colab import userdata

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

In [ ]:
from openai import OpenAI

MODEL = "openai/gpt-4o-mini"
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'connection OK' in exactly two words."}],
)
print("API check:", response.choices[0].message.content)

---

## Two Evaluation Approaches

Imagine your career consultant bot receives "What's the average salary for a data scientist in Berlin?" It should call `compare_salaries` with the right parameters. But what if it calls `search_jobs` instead? Or calls the right tool but passes `"Germany"` instead of `"Berlin"`? Or skips the tool entirely and makes up a number?

Each of these is a different kind of failure, and you need to catch all of them. We will use two complementary approaches:

| Approach | What it checks | How |
|----------|---------------|-----|
| **Deterministic** | Did the agent pick the expected tool and pass the right params? | String comparison (fast, free) |
| **DiscreteMetric** | Was the tool selection *reasonable*? | RAGAs LLM-as-a-judge (nuanced, explains reasoning) |

### Three Levels of Tool Calling Correctness

Tool calling can fail at multiple levels:

| Level | What it checks | How to check |
|-------|---------------|-------------|
| **Tool Selection** | Did the agent pick the right tool? | Deterministic comparison |
| **Parameter Extraction** | Did it pass correct parameters? | Deterministic comparison |
| **Answer Quality** | Is the final response good? | LLM-as-a-judge |

In this notebook we focus on the first two levels (deterministic) and then add LLM-as-a-judge for a more nuanced view of tool selection.

In [ ]:
import json

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent


# Define tools for the career consultant
@tool
def search_jobs(role: str, location: str) -> str:
    """Search for job openings matching a specific role and location."""
    return json.dumps(
        {
            "jobs": [
                {"title": f"Senior {role}", "company": "TechCorp", "location": location},
                {"title": role, "company": "DataInc", "location": location},
            ]
        }
    )


@tool
def compare_salaries(role: str, location: str) -> str:
    """Compare average salaries for a given role in a specific location."""
    return json.dumps(
        {
            "role": role,
            "location": location,
            "average_salary": 78500,
            "currency": "EUR",
            "range": {"min": 65000, "max": 92000},
        }
    )


@tool
def analyze_resume(resume_text: str) -> str:
    """Analyze a resume and provide improvement suggestions."""
    return json.dumps(
        {
            "score": 7,
            "suggestions": [
                "Add more quantifiable achievements",
                "Include relevant certifications",
            ],
        }
    )


# Create agent
tools = [search_jobs, compare_salaries, analyze_resume]
model = ChatOpenAI(model=MODEL, base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a career consultant. Use your tools when you need "
        "real data. For general advice questions, respond directly "
        "without calling tools."
    ),
)
print("Agent created with tools:", [t.name for t in tools])

### Test Dataset Design

A good evaluation dataset covers a range of scenarios with varying difficulty.

| # | Query | Expected Tool | Difficulty |
|---|-------|--------------|------------|
| 1 | "Average salary for data scientist in Berlin?" | `compare_salaries` | Easy |
| 2 | "Find me ML engineer jobs in London" | `search_jobs` | Easy |
| 3 | "Review my resume?" (with text) | `analyze_resume` | Easy |
| 4 | "Thinking about moving to Berlin... I do data science" | `compare_salaries` | Medium |
| 5 | "Friend told me to fix my CV. 3 years Python/ML at Google." | `analyze_resume` | Medium |
| 6 | "Can you search for how much a product manager earns?" | `compare_salaries` | Medium |
| 7 | "What skills should I learn for ML engineering?" | None | Medium |
| 8 | "ML engineer positions and their pay in Amsterdam" | `compare_salaries` | Hard |
| 9 | "Compare the job markets in Berlin and London" | `search_jobs` | Hard |
| 10 | "Resume: Python, SQL, 2 years. What salary should I expect?" | `compare_salaries` | Hard |
| 11 | "Tell me everything about data engineering in Munich" | `search_jobs` | Hard |
| 12 | "I hate my job. Help me." | None | Edge case |
| 13 | "What's hot in the tech job market right now?" | None | Edge case |
| 14 | "Tell me a joke" | None | Edge case |

> The medium and hard queries use indirect phrasing, misleading keywords ("search" when salary is wanted), and blended intents.

In [ ]:
test_cases = [
    # ── Easy: clear intent, obvious tool ──
    {
        "query": "Average salary for data scientist in Berlin?",
        "expected_tool": "compare_salaries",
        "expected_params": {"role": "data scientist", "location": "Berlin"},
        "difficulty": "easy",
    },
    {
        "query": "Find me ML engineer jobs in London",
        "expected_tool": "search_jobs",
        "expected_params": {"role": "ML engineer", "location": "London"},
        "difficulty": "easy",
    },
    {
        "query": (
            "Review my resume? Here it is: "
            "Experienced data scientist with 5 years in ML. "
            "Skills: Python, TensorFlow, SQL. "
            "Previous role: Senior Analyst at DataCorp."
        ),
        "expected_tool": "analyze_resume",
        "expected_params": None,  # free-form text, skip exact param check
        "difficulty": "easy",
    },
    # ── Medium: indirect phrasing, misleading keywords ──
    {
        "query": (
            "Thinking about moving to Berlin, wondering what people "
            "like me make — I do data science"
        ),
        "expected_tool": "compare_salaries",
        "expected_params": None,
        "difficulty": "medium",
    },
    {
        "query": "Friend told me to fix my CV. 3 years Python/ML at Google.",
        "expected_tool": "analyze_resume",
        "expected_params": None,
        "difficulty": "medium",
    },
    {
        "query": "Can you search for how much a product manager earns?",
        "expected_tool": "compare_salaries",
        "expected_params": None,
        "difficulty": "medium",
    },
    {
        "query": "What skills should I learn for ML engineering?",
        "expected_tool": None,
        "expected_params": None,
        "difficulty": "medium",
    },
    # ── Hard: blended intents, multi-tool ambiguity ──
    {
        "query": "Need to know about ML engineer positions and their pay in Amsterdam",
        "expected_tool": "compare_salaries",
        "expected_params": None,
        "difficulty": "hard",
    },
    {
        "query": "Compare the job markets in Berlin and London for data scientists",
        "expected_tool": "search_jobs",
        "expected_params": None,
        "difficulty": "hard",
    },
    {
        "query": "Resume: Python, SQL, 2 years. What salary should I expect?",
        "expected_tool": "compare_salaries",
        "expected_params": None,
        "difficulty": "hard",
    },
    {
        "query": "Tell me everything about data engineering in Munich",
        "expected_tool": "search_jobs",
        "expected_params": None,
        "difficulty": "hard",
    },
    # ── Edge cases: no tool should be called ──
    {
        "query": "I hate my job. Help me.",
        "expected_tool": None,
        "expected_params": None,
        "difficulty": "edge_case",
    },
    {
        "query": "What's hot in the tech job market right now?",
        "expected_tool": None,
        "expected_params": None,
        "difficulty": "edge_case",
    },
    {
        "query": "Tell me a joke",
        "expected_tool": None,
        "expected_params": None,
        "difficulty": "edge_case",
    },
]

print(f"Total test cases: {len(test_cases)}")

from collections import defaultdict

by_difficulty = defaultdict(list)
for case in test_cases:
    by_difficulty[case["difficulty"]].append(case)

for diff, cases in by_difficulty.items():
    print(f"  {diff}: {len(cases)} cases")

In [ ]:
# Run agent on each test case and collect results
results = []

for i, case in enumerate(test_cases):
    print(f"[{i+1}/{len(test_cases)}] {case['query'][:60]}...")

    response = agent.invoke(
        {"messages": [{"role": "user", "content": case["query"]}]}
    )

    # Extract tool calls from all messages in the response
    tool_calls = []
    for msg in response["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tool_calls.extend(msg.tool_calls)

    actual_tool = tool_calls[0]["name"] if tool_calls else None
    actual_params = tool_calls[0]["args"] if tool_calls else None

    # Check tool selection
    tool_correct = actual_tool == case["expected_tool"]

    # Check parameters (only meaningful if tool selection was correct)
    if case["expected_tool"] is None:
        params_correct = tool_correct  # No tool expected = no params to check
    elif case["expected_params"] is None:
        params_correct = tool_correct  # Skip param check for this case
    elif tool_correct and actual_params is not None:
        params_correct = actual_params == case["expected_params"]
    else:
        params_correct = False

    results.append(
        {
            "query": case["query"],
            "expected_tool": case["expected_tool"],
            "actual_tool": actual_tool,
            "tool_correct": tool_correct,
            "expected_params": case["expected_params"],
            "actual_params": actual_params,
            "params_correct": params_correct,
            "difficulty": case["difficulty"],
            "response": response["messages"][-1].content,
        }
    )

print(f"\nCompleted {len(results)}/{len(test_cases)} test cases.")

### Deterministic Evaluation

We compute two metrics:
- **Tool Selection Accuracy** — did the agent pick the right tool (or correctly use no tool)?
- **Parameter Accuracy** — did it pass the correct arguments?

In [ ]:
import pandas as pd

tool_accuracy = sum(r["tool_correct"] for r in results) / len(results)
param_accuracy = sum(r["params_correct"] for r in results) / len(results)

print(f"Tool Selection Accuracy: {tool_accuracy:.0%}")
print(f"Parameter Accuracy:      {param_accuracy:.0%}")
print()

# Breakdown by difficulty
print("Breakdown by difficulty:")
by_difficulty = defaultdict(list)
for r in results:
    by_difficulty[r["difficulty"]].append(r)

for diff, cases in by_difficulty.items():
    acc = sum(c["tool_correct"] for c in cases) / len(cases)
    print(f"  {diff:12s}: {acc:.0%} tool accuracy ({len(cases)} cases)")

# Detailed results
print("\nDetailed results:")
for r in results:
    status = "PASS" if r["tool_correct"] else "FAIL"
    print(f"  [{status}] {r['query'][:55]}...")
    print(f"         Expected: {r['expected_tool']}, Got: {r['actual_tool']}")

### LLM-as-a-Judge with DiscreteMetric

Deterministic checks are strict — they only check if the tool name matches exactly. But sometimes the agent's choice is *reasonable even if different from what we expected*.

**DiscreteMetric** from RAGAs lets us define a custom evaluation prompt and have an LLM judge each case. The judge returns:
- `.value` — a discrete value from the allowed set (e.g. `"correct"` or `"incorrect"`)
- `.reason` — written explanation of why it scored that way

In [ ]:
from ragas.llms import llm_factory
from ragas.metrics import DiscreteMetric

llm = llm_factory(MODEL, client=client)

metric = DiscreteMetric(
    name="tool_selection",
    allowed_values=["correct", "incorrect"],
    prompt=(
        "Evaluate whether the AI agent selected the appropriate tool.\n\n"
        "Available tools:\n"
        "- search_jobs: search for job openings by role and location\n"
        "- compare_salaries: compare salary data for a role in a location\n"
        "- analyze_resume: analyze resume text and suggest improvements\n"
        "- NO TOOL: for general advice, off-topic, or conversational queries\n\n"
        "User query: {user_query}\n"
        "Expected tool: {expected_tool}\n"
        "Actual tool selected: {actual_tool}\n\n"
        "Was the tool selection appropriate for this query?\n"
        "Answer with only 'correct' or 'incorrect'."
    ),
)

print("Scoring each case with DiscreteMetric...\n")
judge_results = []

for i, r in enumerate(results):
    score = metric.score(
        llm=llm,
        user_query=r["query"],
        expected_tool=str(r["expected_tool"]),
        actual_tool=str(r["actual_tool"]),
    )
    judge_results.append({"value": score.value, "reason": score.reason})
    print(f"  Case {i+1}: {score.value} \u2014 {score.reason}")

### Comparing Deterministic vs LLM-as-Judge

The most interesting cases are where the two approaches **disagree**. Deterministic checks are strict (binary match), while DiscreteMetric can recognize that an "incorrect" tool choice was actually reasonable.

In [ ]:
print(f"{'Query':<55} {'Det':>5} {'Judge':>7}")
print("-" * 70)

agree_count = 0
for r, j in zip(results, judge_results):
    det = "PASS" if r["tool_correct"] else "FAIL"
    judge = "PASS" if j["value"] == "correct" else "FAIL"
    marker = "  " if det == judge else "!!"
    if det == judge:
        agree_count += 1

    print(f"{marker} {r['query'][:53]:<53} {det:>5} {judge:>7}")
    if det != judge:
        print(f"     Expected: {r['expected_tool']}, Got: {r['actual_tool']}")
        print(f"     Reason: {j['reason']}")

print(f"\nAgreement: {agree_count}/{len(results)} ({agree_count/len(results):.0%})")
print("!! = disagreement between deterministic and LLM-as-judge")

Note you can try out different models than the `openai/gpt-4o-mini` to observe varying results; for instance, `openai/gpt-5-mini`, `openai/gpt-3.5-turbo`.

Note also that you might observe here that another reason the LLMs are not selecting tools is because the queries are too simple and hence do not deserve the invocation of a tool, from the perspective of an LLM. In that case, if we want to ensure the tools are called even for simple queries, we would have to adjust the agent `system_prompt` defined earlier. On the other hand, it's also important to create _tricky_ queries that best reflect potential failure modes on your applications; the test queries you choose should be selected according to how you would like you agent to behave.

You might observe that the weaker models, such as `openai/gpt-3.5-turbo`, select tools more accurately. This reflects a glance at a novel area of research: the community has discovered that very light _router_ models can be used simply for tool selection, requiring no deep intelligence besides matching the query to a set of tools. A whole class of benchmarks, such as [MetaTool](https://openreview.net/forum?id=R0c2qtalgG), are used for this, and very light models capable of selecting tools as well as the frontier models are used to build more efficient agentic systems.

Tool selection is also evaluated in benchmarks such as 𝜏²-Bench Telecom, Terminal-Bench Hard, GDPval-AA, used by [Artificial Analysis](https://artificialanalysis.ai/methodology/intelligence-benchmarking#artificial-analysis-intelligence-index) to evaluate LLMs.

---

## Summary

In this notebook, you built an evaluation pipeline for tool calling using two complementary approaches:

- **Deterministic checks** — fast, free string comparison for tool selection and parameter accuracy
- **DiscreteMetric (LLM-as-a-judge)** — nuanced evaluation with written reasoning explaining *why* a selection was appropriate or not

### Key Takeaways

- **Start with deterministic checks** (fast, cheap) before adding model-based evaluation
- **Deterministic and LLM-as-judge sometimes disagree** — examining these disagreements reveals where simple checks are too strict or too lenient
- **Include "no tool expected" and edge cases** in every tool calling test dataset
- **Vary difficulty levels** — easy cases validate basic functionality, while hard cases expose where your agent struggles
- **DiscreteMetric's `.reason`** gives you actionable insight into *why* the judge scored each case the way it did